# Day 12 — Aspect-Based Sentiment Analysis (ABSA), Review Topic Modeling & ASO Intelligence Engine

**Project**: Google Play Store Ecosystem Analysis & NLP Autocomplete/Autocorrect Analytics  
**Milestone**: Day 12 (Aspect-Based Sentiment Analysis, LDA Topic Mining, Review Fraud Auditing, ASO Keyword Gap & 360° App Health)  
**Focus**: 5-dimensional operational aspect sentiment extraction, Latent Dirichlet Allocation (LDA) topic discovery, bot brigading / anomaly auditing, competitor ASO keyword gap mining, and 360-degree lifecycle health scorecard.


In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Ensure project root is in sys.path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.aso_intelligence_engine import ASOReviewIntelligenceEngine

# Initialize unified engine
engine = ASOReviewIntelligenceEngine()
print(f'Catalog apps loaded: {len(engine.df_apps):,}')
print(f'Cleaned reviews loaded: {len(engine.df_reviews):,}')


## 1. Aspect-Based Sentiment Analysis (ABSA) across 5 Dimensions

Extract operational aspect mentions across:
- `UI_UX`: Interface, layout, dark mode, aesthetics, navigation
- `PERFORMANCE`: Crashes, lag, slow loading, battery drain, memory
- `MONETIZATION_ADS`: Ads, commercials, subscriptions, paywalls, microtransactions
- `FEATURE_UTILITY`: Tools, tracking, sync, notifications, offline capabilities
- `SUPPORT_RELIABILITY`: Updates, customer service, login, account, privacy

Calculates Net Sentiment Score: $\text{NSS} = (\% \text{Positive} - \% \text{Negative}) \times 100$


In [2]:
# Run aspect-based sentiment assessment for a flagship app
duo_aspects = engine.analyze_aspect_sentiment('Duolingo: Learn Languages Free', is_category=False)

print(f'Target: {duo_aspects.target_name}')
print(f'Dominant Strength: {duo_aspects.dominant_strength.upper()}')
print(f'Dominant Vulnerability: {duo_aspects.dominant_vulnerability.upper()}')
print(f'Aggregate Aspect Health: {duo_aspects.aggregate_aspect_health:.1f}/100\n')

aspect_table = []
for k, score in duo_aspects.aspect_scores.items():
    aspect_table.append({
        'Aspect': k.upper(),
        'Mentions': score.mention_count,
        'Health Index (0-100)': score.aspect_health_index,
        'NSS (%)': f'{score.net_sentiment_score:+.1f}%',
        'Mean Polarity': score.mean_polarity,
        'Keywords': ', '.join(score.representative_keywords[:3])
    })
pd.DataFrame(aspect_table)


## 2. Latent Dirichlet Allocation (LDA) Review Topic Modeling

Discover latent semantic topic clusters across 29,692 real Android app reviews using Scikit-Learn LDA.


In [3]:
# Extract 5 latent review topics
lda_topics = engine.extract_lda_topics(num_topics=5, max_reviews=10000)

topic_table = []
for t in lda_topics:
    topic_table.append({
        'Topic ID': t.topic_id,
        'Thematic Label': t.label,
        'Topic Density (%)': f'{t.topic_weight * 100:.2f}%',
        'Prevalent Sentiment': t.prevalent_sentiment,
        'Top Associative Keywords': ', '.join(t.top_words[:6])
    })
pd.DataFrame(topic_table)


## 3. Review Authenticity & Fraud / Bot Brigading Audit

Statistical audit evaluating:
1. Duplicate text cluster collisions ($Jaccard \ge 0.85$)
2. Polarity-sentiment classification dissonance
3. Ultra-short / repetitive spam anomalies
4. Sentiment entropy index


In [4]:
test_audit_apps = [
    'Duolingo: Learn Languages Free',
    'Subway Surfers',
    'Colorfy: Colouring Book for Adults',
    'Dropbox'
]

audit_table = []
for app_name in test_audit_apps:
    rep = engine.evaluate_review_authenticity(app_name)
    audit_table.append({
        'App Name': rep.app_name,
        'Total Reviews': rep.total_reviews,
        'Authenticity Index (0-100)': rep.authenticity_index,
        'Verdict': rep.verdict,
        'Suspicious Count': rep.suspicious_reviews_count,
        'Duplicates': rep.duplicate_cluster_count,
        'Dissonance': rep.sentiment_dissonance_count,
        'Spam Length': rep.spam_length_count,
        'Entropy Score': rep.entropy_score
    })
pd.DataFrame(audit_table)


## 4. App Store Optimization (ASO) & Competitor Keyword Gap Miner

Mines high-frequency category search keywords used by incumbent market leaders that are missing from target app metadata.


In [5]:
aso_gap = engine.mine_aso_keyword_gap('Duolingo: Learn Languages Free', top_n_keywords=8)

print(f'Target App: {aso_gap.target_app}')
print(f'Category: {aso_gap.category}')
print(f'ASO Optimization Score: {aso_gap.aso_optimization_score:.1f}%')
print(f'Recommended Title Rebrand: {aso_gap.prescribed_title_copy}')
print(f'Current Keywords: {aso_gap.current_keywords}')
print(f'Missing High-Value Category Keywords: {aso_gap.missing_high_value_keywords}\n')

print('Prescribed Copy Directives:')
for bullet in aso_gap.prescribed_feature_bullets:
    print(f'  * {bullet}')


## 5. Holistic 360-Degree App Health Diagnostics & Prescriptive Recourse

Multi-pillar composite index:
$$\text{App Health Index} = 0.30 S_{\text{rating}} + 0.25 S_{\text{sentiment}} + 0.20 S_{\text{aspects}} + 0.15 S_{\text{authenticity}} + 0.10 S_{\text{aso}}$$


In [6]:
health_diag = engine.evaluate_app_health('Duolingo: Learn Languages Free')

print(f'Application: {health_diag.app_name}')
print(f'Category: {health_diag.category} | Rating: {health_diag.rating} | Installs: {health_diag.installs:,}')
print(f'Health Tier: {health_diag.health_tier} | Composite Health Index: {health_diag.composite_health_index:.1f}/100\n')

pillar_table = [{
    'Rating Score (30%)': health_diag.rating_score,
    'Sentiment Score (25%)': health_diag.sentiment_score,
    'Aspect Health (20%)': health_diag.aspect_health_score,
    'Authenticity (15%)': health_diag.authenticity_score,
    'ASO Coverage (10%)': health_diag.aso_optimization_score,
    'Composite Index': health_diag.composite_health_index
}]
print(pd.DataFrame(pillar_table).to_string())

print('\nActionable Prescriptive Recourse Recommendations:')
for p in health_diag.prescriptions:
    print(f'  -> {p}')


## 6. Publication Visualizations & Analytical CSV Reports

Verifies generated Publication Figure 24, Figure 25, and exported summary reports.


In [7]:
# Verify publication figures on disk
fig24_path = project_root / 'reports' / 'figures' / '24_aspect_based_sentiment_and_topic_modeling.png'
fig25_path = project_root / 'reports' / 'figures' / '25_aso_competitor_keyword_gap_and_app_health.png'

print(f'Figure 24 Exists: {fig24_path.exists()} ({fig24_path.stat().st_size / 1024:.1f} KB)')
print(f'Figure 25 Exists: {fig25_path.exists()} ({fig25_path.stat().st_size / 1024:.1f} KB)')

# Check summary reports
reports = [
    'aspect_sentiment_summary.csv',
    'lda_topic_clusters.csv',
    'app_health_scorecard.csv',
    'aso_keyword_opportunity_matrix.csv'
]
for r in reports:
    p = project_root / 'reports' / r
    print(f'Report {r}: {p.exists()} ({p.stat().st_size} bytes)')
